# External Volume vs External Table: When to Use Which?

**External Volume**

- An External Volume is a unity catalog object that provides govered access to files stored in an external cloud storage location
- The files remin in our cloud storage but we access them through the unity catalog volume 
- Way to access or read it -: **/Volumes/dataxdevelopment_1/default/dataxexternalvolume/OrdersFiles/orders.csv**

**External Table**
- An external table is a registered table whoese data files are sotred at an external cloud storage location 
- Unity catalog manages the table metadata and permissions 
- But the external storage location and underlying files remins seperately managed by cloud provider 



In [0]:
dbutils.widgets.text("catalog", "dataxdevelopment_1", "Unity Catalog")
catalog = dbutils.widgets.get("catalog")
if not catalog:
    raise ValueError("Parameter 'catalog' is required")
spark.sql(f"USE CATALOG `{catalog}`")
print(f"Using catalog: {catalog}")

Using catalog: dataxdevelopment_1


In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS ${catalog}.bronze.orders_external
LOCATION 'abfss://dev-datax@dataxazure.dfs.core.windows.net/Volume_External/OrderData'


In [0]:
%sql 
SHOW VOLUMES IN ${catalog}.bronze;

database,volume_name
bronze,orders_external
bronze,orders_files


In [0]:
volume_path =f"/Volumes/{catalog}/bronze/orders_external/"
display(dbutils.fs.ls(volume_path))

path,name,size,modificationTime
dbfs:/Volumes/dataxdevelopment_1/bronze/orders_external/orders.csv,orders.csv,527,1793115591000


In [0]:
order_path=f'/Volumes/{catalog}/bronze/orders_external/orders.csv'

orders_df=spark.read.option('header','true').option('inferSchema','true').csv(order_path)
display(orders_df)

order_id,customer_id,product_id,order_date,quantity,unit_price,order_status
1001,C101,P101,2026-09-01,2,899.0,COMPLETED
1002,C102,P103,2026-09-01,1,1499.0,COMPLETED
1003,C103,P102,2026-09-02,3,499.0,PENDING
1004,C101,P104,2026-09-02,1,2199.0,CANCELLED
1005,C104,P101,2026-09-03,2,899.0,COMPLETED
1006,C105,P105,2026-09-03,1,3499.0,COMPLETED
1007,C102,P103,2026-09-04,2,1499.0,PENDING
1008,C106,P106,2026-09-04,4,299.0,COMPLETED
1009,C107,P102,2026-09-05,1,499.0,COMPLETED
1010,C103,P104,2026-09-05,2,2199.0,COMPLETED


In [0]:
%sql 
CREATE TABLE IF NOT EXISTS ${catalog}.bronze.OrderData_ExternalTable(
   order_id INT,
    customer_id STRING,
    product_id STRING,
    order_date DATE,
    quantity INT,
    unit_price DECIMAL(10,2),
    order_status STRING
)
USING DELTA
LOCATION 'abfss://dev-datax@dataxazure.dfs.core.windows.net/landing_v3/OrderData_ExternalTable'


In [0]:
orders_df.write.mode('append').saveAsTable(f'{catalog}.bronze.OrderData_ExternalTable')

In [0]:
%sql
DROP TABLE ${catalog}.bronze.OrderData_ExternalTable

In [0]:
%sql 
CREATE TABLE IF NOT EXISTS ${catalog}.bronze.OrderData_ExternalTable(
   order_id INT,
    customer_id STRING,
    product_id STRING,
    order_date DATE,
    quantity INT,
    unit_price DECIMAL(10,2),
    order_status STRING
)
USING DELTA
LOCATION 'abfss://dev-datax@dataxazure.dfs.core.windows.net/landing_v3/OrderData_ExternalTable'


In [0]:
%sql
Select * from ${catalog}.bronze.OrderData_ExternalTable

order_id,customer_id,product_id,order_date,quantity,unit_price,order_status
1001,C101,P101,2026-09-01,2,899.00,COMPLETED
1002,C102,P103,2026-09-01,1,1499.00,COMPLETED
1003,C103,P102,2026-09-02,3,499.00,PENDING
1004,C101,P104,2026-09-02,1,2199.00,CANCELLED
1005,C104,P101,2026-09-03,2,899.00,COMPLETED
1006,C105,P105,2026-09-03,1,3499.00,COMPLETED
1007,C102,P103,2026-09-04,2,1499.00,PENDING
1008,C106,P106,2026-09-04,4,299.00,COMPLETED
1009,C107,P102,2026-09-05,1,499.00,COMPLETED
1010,C103,P104,2026-09-05,2,2199.00,COMPLETED


# Production Scenario: When Should We Use External Tables?

- Existing Delta data must remain at a specified cloud storage path.
- Another system or team manages the storage lifecycle.
- An organization has specific storage ownership requirements.
- A migration needs to preserve an existing external dataset location.


# When managed Tables are Better?

- Databricks owns the complete data lifecysle 
- WE want databricks managed storage and optimizations 
- we dont have a requirement to control the phycical table path
